Expected classement de la Pro D2 saison 26//27

In [97]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
import matplotlib.pyplot as plt
import seaborn as sns
import random
from google.colab import drive
drive.mount('/content/drive')
url27 = "https://raw.githubusercontent.com/transientlunatic/Rugby-Data/refs/heads/master/json/pro-d2-2026-2027.json"
df27 = pd.read_json(url27)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [98]:
# 1.CLeaner
df = df27.copy()

# --- A. EXTRACTION DES DONNÉES ---
def extraire_infos(cellule):
    if not isinstance(cellule, dict):
        return pd.Series([None, 0, 0])
    equipe = cellule.get('team')
    score = cellule.get('score', 0)
    essais = 0
    if 'scores' in cellule and isinstance(cellule['scores'], list):
        for action in cellule['scores']:
            if action.get('type') in ['Try', 'Penalty Try']:
                essais += 1
    return pd.Series([equipe, score, essais])

# Extraction
df[['equipe_domicile', 'home_score', 'home_tries']] = df['home'].apply(extraire_infos)
df[['equipe_exterieur', 'away_score', 'away_tries']] = df['away'].apply(extraire_infos)

# --- B. CALCUL DES POINTS (PRO D2) ---
def calculer_points_pro_d2(row):
    if pd.isna(row['home_score']) or pd.isna(row['away_score']):
        return pd.Series([0, 0, 0, 0, 0, 0])

    score_dom = int(row['home_score'])
    score_ext = int(row['away_score'])
    essais_dom = int(row['home_tries'])
    essais_ext = int(row['away_tries'])

    pts_dom, pts_ext = 0, 0
    bonus_off_dom, bonus_off_ext = 0, 0
    bonus_def_dom, bonus_def_ext = 0, 0

    if score_dom > score_ext: pts_dom = 4
    elif score_dom < score_ext: pts_ext = 4
    else: pts_dom = 2; pts_ext = 2

    if essais_dom - essais_ext >= 3: bonus_off_dom = 1
    if essais_ext - essais_dom >= 3: bonus_off_ext = 1

    ecart = abs(score_dom - score_ext)
    if ecart <= 5:
        if score_dom < score_ext: bonus_def_dom = 1
        elif score_ext < score_dom: bonus_def_ext = 1

    total_dom = pts_dom + bonus_off_dom + bonus_def_dom
    total_ext = pts_ext + bonus_off_ext + bonus_def_ext

    return pd.Series([total_dom, total_ext, bonus_off_dom, bonus_off_ext, bonus_def_dom, bonus_def_ext])

# Application
colonnes_calculs = ['pts_domicile', 'pts_exterieur', 'bonus_off_dom', 'bonus_off_ext', 'bonus_def_dom', 'bonus_def_ext']
df[colonnes_calculs] = df.apply(calculer_points_pro_d2, axis=1)

# --- C. CLASSEMENT RÉEL ---
dom = df[['equipe_domicile', 'pts_domicile']].copy()
dom.columns = ['Equipe', 'Points']
ext = df[['equipe_exterieur', 'pts_exterieur']].copy()
ext.columns = ['Equipe', 'Points']

classement_reel = pd.concat([dom, ext]).groupby('Equipe')['Points'].sum().reset_index()
classement_reel = classement_reel.sort_values(by='Points', ascending=False).reset_index(drop=True)
classement_reel['Rang_Reel'] = classement_reel.index + 1
classement_reel.head(16)

,Equipe,Points,Rang_Reel
0,Oyonnax,34,1
1,Provence,34,2
2,Grenoble,26,3
3,Soyaux Angouleme,22,4
4,Montauban,21,5
5,Aurillac,21,6
6,Brive,21,7
7,Biarritz,20,8
8,Agen,19,9
9,Dax,19,10


In [99]:
# ==========================================
# 1. PRÉPARATION DES DONNÉES
# ==========================================

df_N1 = pd.read_csv('/content/drive/MyDrive/Rugby_ProD2_Forecast/matchs_N-1.csv')
df_N2 = pd.read_csv('/content/drive/MyDrive/Rugby_ProD2_Forecast/matchs_N-2.csv')


df_actuel_joue = df.dropna(subset=['home_score']).copy()
df_actuel_futur = df[df['home_score'].isna()].copy()

# On récupère la liste EXACTE des 16 équipes de cette saison
equipes_actuelles = list(set(df_actuel_joue['equipe_domicile'].unique().tolist() + df_actuel_joue['equipe_exterieur'].unique().tolist()))

print(f"Matchs déjà joués en 26/27 : {len(df_actuel_joue)}")
print(f"Matchs à simuler : {len(df_actuel_futur)}")
print(f"Nombre d'équipes en cours : {len(equipes_actuelles)}")



Matchs déjà joués en 26/27 : 64
Matchs à simuler : 136
Nombre d'équipes en cours : 16


In [100]:
# ==========================================
# 2. FONCTION DE CALCUL DES FORCES
# ==========================================
def calculer_forces(df, nom_saison):
    if len(df) == 0:
        return pd.DataFrame(columns=['Equipe', 'Attaque', 'Defense']), 20.0

    df_joues = df.dropna(subset=['home_score'])
    moyenne_generale = df_joues[['home_score', 'away_score']].mean().mean()

    stats_dom = df_joues.groupby('equipe_domicile').agg(
        marques_dom=('home_score', 'mean'), encaisses_dom=('away_score', 'mean')
    ).reset_index()

    stats_ext = df_joues.groupby('equipe_exterieur').agg(
        marques_ext=('away_score', 'mean'), encaisses_ext=('home_score', 'mean')
    ).reset_index()

    stats = pd.merge(stats_dom, stats_ext, left_on='equipe_domicile', right_on='equipe_exterieur', how='outer')
    stats['Equipe'] = stats['equipe_domicile'].fillna(stats['equipe_exterieur'])
    stats = stats.drop(columns=['equipe_domicile', 'equipe_exterieur'])

    stats['Attaque'] = ((stats['marques_dom'] + stats['marques_ext']) / 2).fillna(moyenne_generale)
    stats['Defense'] = ((stats['encaisses_dom'] + stats['encaisses_ext']) / 2).fillna(moyenne_generale)

    return stats, moyenne_generale


In [101]:
# ==========================================
# 3. CALCUL DES FORCES POUR LES 3 SAISONS
# ==========================================
stats_N2, moyenne_N2 = calculer_forces(df_N2, "N-2")
stats_N1, moyenne_N1 = calculer_forces(df_N1, "N-1")
stats_actuel, moyenne_actuelle = calculer_forces(df_actuel_joue, "Actuel")

# On renomme les colonnes pour éviter les conflits
stats_actuel = stats_actuel.rename(columns={'Attaque': 'Attaque_actuel', 'Defense': 'Defense_actuel'})
stats_N1 = stats_N1.rename(columns={'Attaque': 'Attaque_N1', 'Defense': 'Defense_N1'})
stats_N2 = stats_N2.rename(columns={'Attaque': 'Attaque_N2', 'Defense': 'Defense_N2'})

# On fusionne
forces = pd.merge(stats_actuel, stats_N1, on='Equipe', how='outer')
forces = pd.merge(forces, stats_N2, on='Equipe', how='outer')

# On remplit les valeurs manquantes
for col in ['Attaque_actuel', 'Defense_actuel', 'Attaque_N1', 'Defense_N1', 'Attaque_N2', 'Defense_N2']:
    forces[col] = forces[col].fillna(moyenne_actuelle)

In [102]:
# ==========================================
# 4. MÉLANGE DES FORCES (Pondération)
# ==========================================
nombre_matchs_actuels = len(df_actuel_joue) / 16

if nombre_matchs_actuels < 5:
    poids_actuel, poids_n1, poids_n2 = 0.5, 1.0, 0.5
else:
    poids_actuel, poids_n1, poids_n2 = 1.0, 0.4, 0.1

print(f"Poids appliqués -> Actuel: {poids_actuel}, N-1: {poids_n1}, N-2: {poids_n2}")

forces['Attaque_Finale'] = (
    forces['Attaque_actuel'] * poids_actuel +
    forces['Attaque_N1'] * poids_n1 +
    forces['Attaque_N2'] * poids_n2
) / (poids_actuel + poids_n1 + poids_n2)

forces['Defense_Finale'] = (
    forces['Defense_actuel'] * poids_actuel +
    forces['Defense_N1'] * poids_n1 +
    forces['Defense_N2'] * poids_n2
) / (poids_actuel + poids_n1 + poids_n2)

Poids appliqués -> Actuel: 0.5, N-1: 1.0, N-2: 0.5


In [103]:
# ==========================================
# 5. NORMALISATION
# ==========================================
moyenne_attaques = forces['Attaque_Finale'].mean()
moyenne_defenses = forces['Defense_Finale'].mean()
moyenne_ref = moyenne_N1

forces['Attaque_Finale'] = forces['Attaque_Finale'] * (moyenne_ref / moyenne_attaques)
forces['Defense_Finale'] = forces['Defense_Finale'] * (moyenne_ref / moyenne_defenses)

In [95]:
# ==========================================
# 6. SIMULATION MONTE-CARLO
# ==========================================
forces_dict = {}
for _, row in forces.iterrows():
    # 🔥 CORRECTION 2 : On ne garde que les équipes qui jouent cette année
    if row['Equipe'] in equipes_actuelles:
        attaque = max(row['Attaque_Finale'], 0.1) if not pd.isna(row['Attaque_Finale']) else 15.0
        defense = max(row['Defense_Finale'], 0.1) if not pd.isna(row['Defense_Finale']) else 15.0
        forces_dict[row['Equipe']] = {'attaque': attaque, 'defense': defense}

nombre_simulations = 1000
equipes = list(forces_dict.keys())
matchs_infos = []
equipe_index = {equipe: i for i, equipe in enumerate(equipes)}

for _, row in df_actuel_futur.iterrows():
    dom = row['equipe_domicile']
    ext = row['equipe_exterieur']
    if dom in forces_dict and ext in forces_dict:
        lambda_dom = (forces_dict[dom]['attaque'] * forces_dict[ext]['defense']) / moyenne_ref
        lambda_ext = (forces_dict[ext]['attaque'] * forces_dict[dom]['defense']) / moyenne_ref
        lambda_dom = min(max(lambda_dom, 0.1), 50)
        lambda_ext = min(max(lambda_ext, 0.1), 50)
        matchs_infos.append((equipe_index[dom], equipe_index[ext], lambda_dom, lambda_ext))

resultats = np.zeros((nombre_simulations, len(equipes)))

for sim in range(nombre_simulations):
    points_saison = np.zeros(len(equipes))
    for idx_dom, idx_ext, lambda_dom, lambda_ext in matchs_infos:
        score_dom = poisson.rvs(lambda_dom)
        score_ext = poisson.rvs(lambda_ext)

        if score_dom > score_ext: points_saison[idx_dom] += 4
        elif score_dom < score_ext: points_saison[idx_ext] += 4
        else:
            points_saison[idx_dom] += 2
            points_saison[idx_ext] += 2

        if abs(score_dom - score_ext) <= 5 and score_dom != score_ext:
            if score_dom < score_ext: points_saison[idx_dom] += 1
            else: points_saison[idx_ext] += 1
    resultats[sim] = points_saison


In [96]:
# ==========================================
# 7. RÉSULTATS FINAUX
# ==========================================
# Points déjà acquis
dom_pts = df_actuel_joue.groupby('equipe_domicile')['pts_domicile'].sum().reset_index()
dom_pts.columns = ['Equipe', 'Points']
ext_pts = df_actuel_joue.groupby('equipe_exterieur')['pts_exterieur'].sum().reset_index()
ext_pts.columns = ['Equipe', 'Points']
points_deja_acquis = pd.concat([dom_pts, ext_pts]).groupby('Equipe')['Points'].sum()

# On calcule la moyenne des points simulés
moyennes_points = np.mean(resultats, axis=0) + points_deja_acquis.reindex(equipes).fillna(0).values

df_final = pd.DataFrame({'Equipe': equipes, 'Expected_Points': moyennes_points})
df_final = df_final.sort_values(by='Expected_Points', ascending=False).reset_index(drop=True)
df_final['Expected_Rank'] = df_final.index + 1

# 🔥 CORRECTION 3 : On s'assure que le tableau final ne contient que les 16 équipes
df_final = df_final[df_final['Equipe'].isin(equipes_actuelles)].reset_index(drop=True)
df_final['Expected_Rank'] = df_final.index + 1

print("\n CLASSEMENT ATTENDU FINAL - PRO D2 2026/2027 :")
display(df_final)


 CLASSEMENT ATTENDU FINAL - PRO D2 2026/2027 :


,Equipe,Expected_Points,Expected_Rank
0,Oyonnax,93.907,1
1,Provence,89.887,2
2,Brive,74.786,3
3,Grenoble,74.314,4
4,Colomiers,70.752,5
5,Valence-Romans,61.889,6
6,Agen,58.708,7
7,Soyaux Angouleme,55.468,8
8,Montauban,54.377,9
9,Biarritz,50.793,10


In [105]:
df_final.to_csv('/content/drive/MyDrive/Rugby_ProD2_Forecast/expected_rank_j8.csv', index=False)